# Earthcare_Quicklooks_Scraping
A Google Gemini dialog set this up for me. 

"Operating the site via the "Next" button is a much more robust way to handle the varying number of orbits per day. Playwright can handle this beautifully by using a loop that clicks the button and then waits for the page content to update before capturing the PDF.

The Plan
Launch & Manual Setup: Use headless=False initially so the browser window pops up. This allows you to manually uncheck the unwanted boxes on the first page.

The Loop:

Find the "Next" button element (likely an <a> or <button> tag).

Capture the current view as a PDF page.

Click "Next" and wait for the new quicklook image to load.

PDF Compilation: Use page.pdf() to generate a high-quality capture. To merge 365+ captures into a single document, we can use PyPDF2.

Python Implementation
You'll need pip install playwright pypdf2 followed by playwright install chromium."

Implementation Notes for your Workflow:
The "Prev" Selector: Japanese sites often use specific alt-text for navigation arrows. If page.get_by_text("Prev") doesn't trigger, look at the button in the browser: if it's an image, we might need page.locator("img[alt='Prev']").click().

Scale Factor: I added scale=0.8 to the page.pdf command. Often, browser headers and footers cut off the bottom of the data on a standard A4 sheet; shrinking it slightly usually ensures the full "Quicklook" plot fits.

State Persistence: Because we are using the same page object throughout the loop, the boxes you uncheck at the start should stay unchecked as you click "Prev," provided the site uses standard session/URL state behavior.

Memory: By writing the PdfWriter to a file every 100 pages and starting a fresh one, we keep the Jupyter kernel's memory usage low.

In [13]:
# The code to make it all happen! 
# works, but pixel jitter is annoying 

import asyncio
import io
from playwright.async_api import async_playwright
from PyPDF2 import PdfWriter

async def archive_earthcare_backwards(total_days=365, pages_per_file=100):
    async with async_playwright() as p:
        # 1. Launch visible browser for manual setup
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()

        # Start URL (your provided latest date)
        url = "https://www.eorc.jaxa.jp/EARTHCARE/Quicklook/index_j.html?date=20260212&orbit=09723&frame=E"
        await page.goto(url)

        # 2. Manual Intervention Step
        print("-" * 30)
        print("ACTION REQUIRED:")
        print("1. Uncheck the unwanted boxes in the browser window.")
        print("2. Scroll or resize if needed.")
        print("3. Return here and press ENTER to start.")
        print("-" * 30)
        input("Press Enter to begin automation...")

        captured_count = 0
        file_index = 1
        
        while captured_count < total_days:
            writer = PdfWriter()
            current_chunk_size = min(pages_per_file, total_days - captured_count)
            filename = f"EarthCare_Archive.pdf"
            
            print(f"\nStarting {filename}...")

            for i in range(current_chunk_size):
                # Capture current view
                # scale=0.8 can help fit more content onto a standard A4 page
                pdf_bytes = await page.pdf(format="A4", print_background=True, scale=0.8)
                writer.append(io.BytesIO(pdf_bytes))
                                
                # 3. Navigate to "Previous"
                try:
                    # Selector: Searches for the 'Prev' or '前へ' link/button
                    # If this fails, we can swap to page.mouse.click(x, y)
                    prev_button = page.get_by_text("Prev") 
                    await prev_button.click()
                    
                    # Wait for the image/data to flip
                    await page.wait_for_load_state('networkidle')
                    await asyncio.sleep(3) # Short pause for stability - 0.5s was too short!
                except Exception as e:
                    print(f"Navigation stopped: {e}")
                    break
                
                captured_count += 1
                if captured_count % 10 == 0:
                    print(f"Progress: {captured_count}/{total_days} orbits captured.")

            # Save the chunk
            with open(filename, "wb") as f:
                writer.write(f)
            print(f"Saved {filename}")
            file_index += 1

        await browser.close()
        print("\nAll tasks complete.")

In [15]:
# To run in your Jupyter cell:
await archive_earthcare_backwards(total_days=190, pages_per_file=190)

------------------------------
ACTION REQUIRED:
1. Uncheck the unwanted boxes in the browser window.
2. Scroll or resize if needed.
3. Return here and press ENTER to start.
------------------------------


Press Enter to begin automation... 



Starting EarthCare_Archive.pdf...
Progress: 10/190 orbits captured.
Progress: 20/190 orbits captured.
Progress: 30/190 orbits captured.
Progress: 40/190 orbits captured.
Progress: 50/190 orbits captured.
Progress: 60/190 orbits captured.
Progress: 70/190 orbits captured.
Progress: 80/190 orbits captured.
Progress: 90/190 orbits captured.
Progress: 100/190 orbits captured.
Progress: 110/190 orbits captured.
Progress: 120/190 orbits captured.
Progress: 130/190 orbits captured.
Progress: 140/190 orbits captured.
Progress: 150/190 orbits captured.
Progress: 160/190 orbits captured.
Progress: 170/190 orbits captured.
Progress: 180/190 orbits captured.
Progress: 190/190 orbits captured.
Saved EarthCare_Archive.pdf

All tasks complete.


# Blow up the lidar vertically 
import fitz; print(fitz.open("EarthCare_Quicklooks_SEAtl.pdf")[0].rect) # then Preview Inspector

In [16]:
# bloom (vertically stretch) lidar's lowest levels
import fitz

def bloom_vertical_zoom(input_pdf, output_pdf, y_bottom, height_of_strip):
    target_doc = fitz.open(input_pdf)
    source_doc = fitz.open(input_pdf)
    
    y_top = y_bottom - height_of_strip
    zoomed_height = height_of_strip * 4
    
    for page_index in range(len(target_doc)):
        target_page = target_doc[page_index]
        
        # What to grab
        source_rect = fitz.Rect(0, y_top, target_page.rect.width, y_bottom)
        
        # Where to put it (The 4x taller box)
        target_rect = fitz.Rect(0, y_bottom - zoomed_height, target_page.rect.width, y_bottom)
        
        # show_pdf_page stretches the 'clip' to fit the 'target_rect'
        target_page.show_pdf_page(
            target_rect,
            source_doc,
            page_index,
            clip=source_rect,
            keep_proportion=False  # THIS IS THE KEY: Forces vertical stretching
        )

    target_doc.save(output_pdf)
    target_doc.close()
    source_doc.close()

In [18]:
# Apply to my files: ascending, descending, after setting filenames (verifying it)

# Your measured coordinates # measured by Preview Inspector
height_of_strip = 19.93
y_bottom = 240.21 + height_of_strip 

input_pdf = "EarthCare_Quicklooks_SEAtl_nite.pdf"
output_pdf = "EarthCare_Quicklooks_SEAtl_nite_BLzoom.pdf"
bloom_vertical_zoom(input_pdf, output_pdf, y_bottom, height_of_strip)

input_pdf = "EarthCare_Quicklooks_SEPac_nite.pdf"
output_pdf = "EarthCare_Quicklooks_SEPac_nite_BLzoom.pdf"
bloom_vertical_zoom(input_pdf, output_pdf, y_bottom, height_of_strip)

